# AI Study Assistant on Google Colab

Runs the app on a free Colab GPU, reading your notes from Google Drive.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (a GPU makes indexing take seconds instead of minutes and lets the app use the 4-bit Mistral model).
Then run the cells top to bottom (or *Runtime → Run all*). The last cell prints a public link to open the app.

Your Drive copy of the project is **not modified**: the notebook clones a fresh copy of the code into `/content` and only reads your notes from Drive.

In [ ]:
#@title 1. Settings { display-mode: "form" }
REPO_URL = "https://github.com/ashpakshaikh26732/AI-Study-Assistant.git" #@param {type:"string"}
BRANCH = "feature/cpu-friendly-dashboard" #@param {type:"string"}
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/AI-Study-Assistant" #@param {type:"string"}
PROCESSED_SUBDIR = "processed" #@param {type:"string"}
LLM_PROVIDER = "auto" #@param ["auto", "transformers", "gemini", "none"]
# PROCESSED_SUBDIR = the folder inside <DRIVE_PROJECT_DIR>/data that holds your cleaned .txt notes
#   (use "processed_new" if that's where your reviewed notes live).
# LLM_PROVIDER: auto = GPU 4-bit Mistral if a GPU is attached; gemini = Google API (add a Colab secret GOOGLE_API_KEY);
#   none = no language model (chat just shows the best-matching passages).

In [ ]:
#@title 2. Mount Drive and get the code { display-mode: "form" }
from google.colab import drive
drive.mount("/content/drive")

import os
if not os.path.exists("/content/AI-Study-Assistant"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/AI-Study-Assistant
else:
    !cd /content/AI-Study-Assistant && git fetch -q && git checkout -q {BRANCH} && git pull -q
%cd /content/AI-Study-Assistant
!git log --oneline -1

In [ ]:
#@title 3. Install dependencies (about 2-3 minutes the first time) { display-mode: "form" }
!pip install -q -r requirements.txt
import torch
GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none - the app will still work, just slower to index")
if GPU:
    !pip install -q -r requirements-gpu.txt
!apt-get -qq install -y tesseract-ocr > /dev/null   # only used to OCR scanned PDFs

In [ ]:
#@title 4. Point the app at your Drive data { display-mode: "form" }
import os, pathlib

DATA_DIR = f"{DRIVE_PROJECT_DIR}/data"
os.environ["STUDY_ASSISTANT_DATA_DIR"] = DATA_DIR              # raw notes, processed notes, quiz history (memory.db)
os.environ["STUDY_ASSISTANT_INDEX_DIR"] = "/content/index"      # vector index lives on fast local disk
os.environ["STUDY_ASSISTANT_LLM_PROVIDER"] = LLM_PROVIDER
pathlib.Path("config.local.yaml").write_text(f"data:\n  processed_path: {DATA_DIR}/{PROCESSED_SUBDIR}\n")

n_notes = sum(1 for _ in pathlib.Path(f"{DATA_DIR}/{PROCESSED_SUBDIR}").rglob("*.txt"))
print(f"Found {n_notes} processed note files in {DATA_DIR}/{PROCESSED_SUBDIR}")
if n_notes == 0:
    print("-> Nothing to index. Check PROCESSED_SUBDIR above, or run `!python run_preprocessing.py` to convert your raw PDFs.")

if LLM_PROVIDER == "gemini":
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")   # add it under the key icon in the left sidebar
elif LLM_PROVIDER in ("auto", "transformers") and GPU:
    # Mistral-7B-Instruct is a gated model: accept its terms once at
    # https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3 then paste an access token below.
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
#@title 5. Build / update the index { display-mode: "form" }
# The index is backed up to Drive, so later sessions restore it and only embed new or changed notes.
import tarfile
BACKUP = f"{DRIVE_PROJECT_DIR}/index_backup.tar.gz"
if os.path.exists(BACKUP) and not os.path.exists("/content/index/manifest.json"):
    print("Restoring the saved index from Drive...")
    with tarfile.open(BACKUP) as tar:
        tar.extractall("/content")

!python build_vector_store.py

with tarfile.open(BACKUP, "w:gz") as tar:
    tar.add("/content/index", arcname="index")
print("Index backed up to", BACKUP)

In [ ]:
#@title 6. Launch the app and get a public link { display-mode: "form" }
import re, subprocess, threading, time

subprocess.Popen(
    ["streamlit", "run", "src/app/main.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT,
)
time.sleep(6)

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stderr=subprocess.PIPE, text=True)
url = None
for line in tunnel.stderr:
    match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stderr], daemon=True).start()   # keep the pipe drained
print("\nOpen the app here:", url or "(no link found - check the cloudflared output above)")
print("Keep this cell running; stopping it (or the runtime) closes the app.")

### Notes
* **First question is slow, later ones are fast**: the embedding model and the language model load on first use.
* Your quiz history is saved to Drive (`data/memory.db`), so your progress carries across sessions.
* Want to add notes? Use the app's **Library → Add notes** page, or drop files in Drive and re-run step 5.
* Using ngrok instead of cloudflared? Replace step 6's tunnel with `pyngrok` and your own auth token.